# Credit Card Fraud Detection — Exploratory Data Analysis

**Dataset:** Kaggle Credit Card Fraud Detection  
**Records:** 284,807 transactions | **Frauds:** 492 (0.172%)  
**Features:** `Time`, `Amount`, `V1`–`V28` (PCA-transformed), `Class` (target)

### Goals
1. Understand the class imbalance and why accuracy is a useless metric here
2. See how fraud transactions differ in `Amount` and `Time`
3. Identify which V-features are most discriminative
4. Confirm the engineered features add signal

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure feature_engineering is importable
sys.path.insert(0, str(Path('../../airflow/plugins').resolve()))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path('../../data/raw/creditcard.csv')
assert DATA_PATH.exists(), f'Run `make download-data` first. Expected: {DATA_PATH.resolve()}'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Class Distribution

The core challenge: fraud is a *rare event*. A model that always predicts "not fraud" achieves 99.83% accuracy — useless in practice. We evaluate with **precision, recall, F1, and PR-AUC** instead.

In [ ]:
counts = df['Class'].value_counts()
pct = df['Class'].value_counts(normalize=True) * 100

print('Class distribution:')
print(f'  Legitimate (0): {counts[0]:>7,}  ({pct[0]:.3f}%)')
print(f'  Fraud      (1): {counts[1]:>7,}  ({pct[1]:.3f}%)')
print(f'  Imbalance ratio: {counts[0] / counts[1]:.0f}:1')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart (raw counts)
axes[0].bar(['Legitimate', 'Fraud'], [counts[0], counts[1]],
            color=['steelblue', 'tomato'])
axes[0].set_title('Raw counts')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)

# Pie chart (proportions)
axes[1].pie([counts[0], counts[1]], labels=['Legitimate', 'Fraud'],
            colors=['steelblue', 'tomato'],
            autopct='%1.3f%%', startangle=140)
axes[1].set_title('Class proportions')

plt.suptitle('Class Distribution — Severe Imbalance', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Transaction Amount

Fraud transactions tend to cluster at **lower amounts** — fraudsters often test stolen cards with small purchases. The distribution is also heavily right-skewed, motivating our `log1p` transform.

In [ ]:
legit = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

print('Amount statistics by class:')
print(pd.DataFrame({'Legitimate': legit.describe(), 'Fraud': fraud.describe()}).round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Raw distributions (clipped to 99th percentile to suppress outliers)
clip = df['Amount'].quantile(0.99)
axes[0].hist(legit.clip(upper=clip), bins=60, alpha=0.6, color='steelblue', label='Legitimate', density=True)
axes[0].hist(fraud.clip(upper=clip), bins=60, alpha=0.6, color='tomato', label='Fraud', density=True)
axes[0].set_title('Amount (clipped at 99th pct)')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

# Log-transformed
axes[1].hist(np.log1p(legit), bins=60, alpha=0.6, color='steelblue', label='Legitimate', density=True)
axes[1].hist(np.log1p(fraud), bins=60, alpha=0.6, color='tomato', label='Fraud', density=True)
axes[1].set_title('log1p(Amount) — reduces skew')
axes[1].set_xlabel('log1p(Amount)')
axes[1].legend()

# Box plot
axes[2].boxplot([np.log1p(legit), np.log1p(fraud)],
                labels=['Legitimate', 'Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].set_title('log1p(Amount) — box plot')
axes[2].set_ylabel('log1p(Amount)')

plt.suptitle('Transaction Amount by Class', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Time of Day

`Time` is seconds elapsed since the first transaction. We convert it to an approximate **hour of day**. Fraud activity may cluster at night (low human oversight).

In [ ]:
df['hour_of_day'] = (df['Time'] // 3600 % 24).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Fraud rate by hour
fraud_rate = df.groupby('hour_of_day')['Class'].mean() * 100
axes[0].bar(fraud_rate.index, fraud_rate.values, color='tomato', alpha=0.8)
axes[0].axhline(df['Class'].mean() * 100, color='gray', linestyle='--', label='Overall rate')
axes[0].set_xlabel('Hour of day (approx.)')
axes[0].set_ylabel('Fraud rate (%)')
axes[0].set_title('Fraud rate by hour')
axes[0].legend()

# Volume by hour (stacked)
hour_counts = df.groupby(['hour_of_day', 'Class']).size().unstack(fill_value=0)
hour_counts.plot(kind='bar', stacked=True, ax=axes[1],
                 color=['steelblue', 'tomato'], alpha=0.8)
axes[1].set_xlabel('Hour of day (approx.)')
axes[1].set_ylabel('Transaction count')
axes[1].set_title('Transaction volume by hour')
axes[1].legend(['Legitimate', 'Fraud'])
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Time Patterns', fontsize=13)
plt.tight_layout()
plt.show()

## 4. V-Feature Discrimination

The 28 PCA components are anonymized but many separate fraud from legitimate transactions well. We rank them by the **absolute mean difference** between classes (a simple but effective signal indicator).

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]

means_legit = df[df['Class'] == 0][v_cols].mean()
means_fraud = df[df['Class'] == 1][v_cols].mean()
discrimination = (means_fraud - means_legit).abs().sort_values(ascending=False)

print('Top 10 most discriminative V-features (|mean_fraud - mean_legit|):')
print(discrimination.head(10).round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['tomato' if v in discrimination.head(5).index else 'steelblue' for v in discrimination.index]
ax.bar(discrimination.index, discrimination.values, color=colors, alpha=0.85)
ax.set_xlabel('V-feature')
ax.set_ylabel('|mean_fraud − mean_legit|')
ax.set_title('V-Feature Discrimination Strength (top 5 highlighted)')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots for the top 6 most discriminative features
top_features = discrimination.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.flat, top_features):
    data_plot = [
        df[df['Class'] == 0][col].values,
        df[df['Class'] == 1][col].values,
    ]
    parts = ax.violinplot(data_plot, showmedians=True)
    for pc, color in zip(parts['bodies'], ['steelblue', 'tomato']):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Legit', 'Fraud'])
    ax.set_title(col)

plt.suptitle('Top 6 Discriminative V-Features (class distributions)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Correlation with Target

Pearson correlation of each feature with `Class`. Note: correlation is weak for imbalanced targets, but it confirms which features are most linearly related to fraud.

In [ ]:
corr_with_class = df.drop(columns=['Class']).corrwith(df['Class']).sort_values()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with Class')
ax.set_title('Feature Correlation with Fraud (Class=1)')
plt.tight_layout()
plt.show()

print('\nStrongest positive correlations with fraud:')
print(corr_with_class.tail(5).round(4).to_string())
print('\nStrongest negative correlations with fraud:')
print(corr_with_class.head(5).round(4).to_string())

## 6. Engineered Features

Verify the five new features add signal on top of the raw data.

In [ ]:
from feature_engineering import engineer_features

df_eng = engineer_features(df)
new_cols = ['amount_log', 'amount_zscore', 'hour_of_day', 'is_night', 'v1_v2_interaction']

print('Engineered features added:', new_cols)
print(f'Shape before: {df.shape}  →  after: {df_eng.shape}')
print()

# Compare means by class for new features
summary = df_eng.groupby('Class')[new_cols].mean().T
summary.columns = ['Legitimate', 'Fraud']
summary['Difference'] = (summary['Fraud'] - summary['Legitimate']).abs()
print(summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# amount_log by class
for cls, color, label in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    axes[0].hist(df_eng[df_eng['Class'] == cls]['amount_log'],
                 bins=50, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('amount_log by class')
axes[0].legend()

# is_night fraud rate
night_rate = df_eng.groupby('is_night')['Class'].mean() * 100
axes[1].bar(['Day', 'Night'], night_rate.values, color=['steelblue', 'tomato'], alpha=0.8)
axes[1].set_title('Fraud rate: day vs night')
axes[1].set_ylabel('Fraud rate (%)')

# v1_v2_interaction by class
clip_val = df_eng['v1_v2_interaction'].quantile(0.99)
for cls, color, label in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    vals = df_eng[df_eng['Class'] == cls]['v1_v2_interaction'].clip(-clip_val, clip_val)
    axes[2].hist(vals, bins=60, alpha=0.6, color=color, label=label, density=True)
axes[2].set_title('V1×V2 interaction by class')
axes[2].legend()

plt.suptitle('Engineered Features', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Finding | Implication |
|---------|-------------|
| 492 frauds / 284,807 total (0.172%) | Use PR-AUC, not accuracy. Apply SMOTE + `scale_pos_weight`. |
| Fraud amounts cluster lower | `amount_log` and `amount_zscore` are useful features. |
| V14, V17, V12 most discriminative | XGBoost will weight these heavily; SHAP will confirm. |
| Night flag shows slightly higher fraud rate | `is_night` adds modest signal. |
| V1×V2 interaction differs by class | Interaction term is worth including. |

**Next step:** Phase 3 — train XGBoost and Autoencoder with MLflow tracking.